<a href="https://colab.research.google.com/github/camistrika/BETO_HUMOR/blob/main/notebooks/cross_validation_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Validación cruzada (k-fold) — BETO baseline (cabeza congelada)
Explora los learning rates más estables vistos en el grid search,
para confirmar con varios folds que el resultado no depende del split particular.

In [1]:
!git clone https://github.com/camistrika/BETO_HUMOR.git
%cd BETO_HUMOR
!pip install -e . -q

Cloning into 'BETO_HUMOR'...
remote: Enumerating objects: 162, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 162 (delta 73), reused 152 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (162/162), 3.23 MiB | 30.35 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/content/BETO_HUMOR
  Preparing metadata (setup.py) ... done


In [2]:
!pip install -q torchao --upgrade
!pip install -q transformers peft datasets scikit-learn pyyaml

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer
from google.colab import files

from betohumor.config import DataConfig, BetoConfig
from betohumor.utils import set_seed
from betohumor.dataset import load_and_split
from betohumor.models.beto import build_beto
from betohumor.hyperparam_search import run_one
from itertools import product

import shutil
import os

In [4]:
%cd /content/BETO_HUMOR

/content/BETO_HUMOR


In [5]:
ls

data/       README.md         results/  smoke_test_curve.png
notebooks/  requirements.txt  setup.py  src/


## 1. Datos y configuraciones

In [ ]:
data_config  = DataConfig(data_path = "data/raw/haha_2019_train.csv")
beto_config = BetoConfig()
set_seed(data_config.seed)

df_train, df_val, df_test = load_and_split(data_config)

df_full = pd.concat([df_train, df_val]).reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained(beto_config.base_model)
label_col = data_config.label_col

## 2. Configuración de la búsqueda
El baseline congelado no usa LoraConfig: la única variable a explorar
es el learning rate.

In [7]:
LR_VALUES = [1e-4, 5e-4, 1e-3]
WEIGHT_DECAYS = [0.01, 0.05]
N_FOLDS = 3

params_grid = [
    {'learning_rate': lr, 'weight_decay': wd}
    for lr, wd in product(LR_VALUES, WEIGHT_DECAYS)
]

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=data_config.seed)
params_grid



[{'learning_rate': 0.0001, 'weight_decay': 0.01},
 {'learning_rate': 0.0001, 'weight_decay': 0.05},
 {'learning_rate': 0.0005, 'weight_decay': 0.01},
 {'learning_rate': 0.0005, 'weight_decay': 0.05},
 {'learning_rate': 0.001, 'weight_decay': 0.01},
 {'learning_rate': 0.001, 'weight_decay': 0.05}]

## 3. Correr la validación cruzada

In [ ]:
all_fold_results = []

for params in params_grid:
    fold_scores = []
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(df_full, df_full[label_col])):
        run_name = f"lr{params['learning_rate']}_wd{params['weight_decay']}_fold{fold_i+1}"
        print(f"\n--- {run_name} ---")

        df_fold_train = df_full.iloc[train_idx].reset_index(drop=True)
        df_fold_val   = df_full.iloc[val_idx].reset_index(drop=True)

        output_dir = f"results/checkpoints/cv_beto/{run_name}"
        macro_f1, trainer = run_one(
            lambda p: build_beto(beto_config),
            params, beto_config, data_config,
            df_fold_train, df_fold_val, tokenizer, output_dir, seed=data_config.seed,
        )

        fold_scores.append(macro_f1)
        all_fold_results.append({**params, 'fold': fold_i + 1, 'macro_f1': macro_f1})
        print(f"Fold {fold_i+1} Macro F1: {macro_f1:.4f}")

        # Borrar checkpoints intermedios para no llenar el disco
        if os.path.exists(output_dir):
            for folder in os.listdir(output_dir):
                if folder.startswith("checkpoint"):
                    shutil.rmtree(os.path.join(output_dir, folder))

    print(f"\n=== lr={params['learning_rate']} wd={params['weight_decay']} -> Media: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f} ===")

## 4. Resumen y guardado

In [9]:
df_folds = pd.DataFrame(all_fold_results)
df_folds.to_csv('results/cv_results_beto.csv', index=False)

df_summary = (
    df_folds
    .groupby(['learning_rate', 'weight_decay'])['macro_f1']
    .agg(mean_macro_f1='mean', std_macro_f1='std')
    .reset_index()
    .sort_values('mean_macro_f1', ascending=False)
)
df_summary.to_csv('results/cv_results_beto_summary.csv', index=False)
df_summary



,learning_rate,weight_decay,mean_macro_f1,std_macro_f1
5,0.0010,0.05,0.841485,0.002632
4,0.0010,0.01,0.841003,0.003257
2,0.0005,0.01,0.840293,0.000957
3,0.0005,0.05,0.839495,0.001702
1,0.0001,0.05,0.833896,0.002234
0,0.0001,0.01,0.832520,0.000430


In [10]:
files.download('results/cv_results_beto.csv')
files.download('results/cv_results_beto_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>